# DR screening models: training on Kaggle

Runs the repository's documented pipeline on a free Kaggle GPU. All logic lives
in the `drscreen` package; this notebook only calls `scripts/`.

**Before running**

1. *Settings → Accelerator*: GPU T4 (recent PyTorch builds may not support the
   older P100). *Settings → Internet*: on (timm downloads ImageNet weights).
2. *Add Input*:
   - the competition **aptos2019-blindness-detection** (accept its rules first);
   - **tanlikesmath/diabetic-retinopathy-resized** (EyePACS training images,
     for grading and the quality model);
   - **benjaminwarner/resized-2015-2019-blindness-detection-images** (EyePACS
     test images and their released grades);
   - **samriddhibagchi/ddr-dataset-credits-to-authors** (DDR);
   - **aaryapatel98/indian-diabetic-retinopathy-image-dataset** (IDRiD, the
     external test set).

   `kagglehub` finds attached inputs, so nothing is downloaded; EyeQ's label
   CSVs come from GitHub.
3. Either attach this repository as a Kaggle dataset (any input containing its
   `pyproject.toml`, ideally with a `GIT_COMMIT` file beside it), or set
   `REPO_URL` below to a copy on GitHub.

The image cache (about 20 GB) goes to `/tmp`, because Kaggle keeps only 20 GB of
`/kaggle/working`. Outputs land in `/kaggle/working/dr-screening/runs/`. Use
*Save Version* to keep them, then download each run's `best.pt` for the web app.


In [ ]:
REPO_URL = "https://github.com/YOUR-USER/dr-screening.git"  # used only if no code input is attached
REPO_REF = "main"
WORK_DIR = "/kaggle/working/dr-screening"

In [ ]:
import shutil
import subprocess
from pathlib import Path

attached = [
    path.parent
    for path in Path("/kaggle/input").glob("**/pyproject.toml")
    if 'name = "drscreen"' in path.read_text()
]
if attached:
    print("Using attached code from", attached[0])
    shutil.copytree(attached[0], WORK_DIR)
    subprocess.run(["chmod", "-R", "u+w", WORK_DIR], check=True)  # inputs are read-only
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, WORK_DIR], check=True
    )
%cd $WORK_DIR

Kaggle's image already provides torch, torchvision, timm, kagglehub, pandas and
scikit-learn built for its GPU. Installing with `--no-deps` adds our package
without replacing them; the next cell checks that every import resolves.

In [ ]:
!pip install --quiet --no-deps -e .
!python -c "import drscreen.training.trainer, torch; print('CUDA:', torch.cuda.is_available())"

## Data: manifests, cropped cache, inspection
A data config pointing every cache at `/tmp` is written first. Preparing about
65,000 photos takes roughly an hour.


In [ ]:
%%bash
python - <<'EOF'
import json
from pathlib import Path
from drscreen.config import load_config
from drscreen.data.settings import DataConfig
root = Path("/tmp/drscreen")
root.mkdir(exist_ok=True)
config = load_config(Path("configs/data.yaml"), DataConfig, [
    f"paths.manifests_dir={root}/manifests",
    f"paths.processed_dir={root}/processed",
    f"paths.reports_dir={root}/reports",
])
(root / "data.json").write_text(json.dumps(config.model_dump(mode="json")))
EOF
python scripts/fetch_data.py --config /tmp/drscreen/data.json
python scripts/prepare_data.py --config /tmp/drscreen/data.json
python scripts/inspect_data.py --config /tmp/drscreen/data.json


In [ ]:
import json

from IPython.display import Image, display

DATA = "/tmp/drscreen/data.json"
for source in ("aptos", "eyepacs", "eyepacs_test", "ddr", "idrid", "eyeq"):
    display(Image(f"/tmp/drscreen/reports/{source}_processed_samples.png"))

## Train the grading model on APTOS, EyePACS and DDR
About 41,000 training photos; 8 epochs of roughly 40 minutes each.


In [ ]:
!python scripts/train.py --config configs/train_grading.yaml --override data_config={DATA}

## Referral threshold, then external validation on IDRiD
The threshold is tuned on the validation photos of all four training sources.
IDRiD was never used for training, model selection or tuning.


In [ ]:
from pathlib import Path

run_dir = max(Path("runs").glob("aptos*"), key=lambda path: path.stat().st_mtime)
checkpoint = run_dir / "best.pt"
print(checkpoint)
!python scripts/tune_referral.py --checkpoint {checkpoint} --override data_config={DATA}
threshold = json.loads((run_dir / "referral_threshold.json").read_text())["threshold"]
evaluate_args = f"--override data_config={DATA} --override referral.threshold={threshold}"
!python scripts/evaluate.py --checkpoint {checkpoint} {evaluate_args}

In [ ]:
report = json.loads((run_dir / "eval-idrid" / "metrics.json").read_text())
for name, value in report["metrics"].items():
    print(f"{name:40s} {value}")
display(Image(str(run_dir / "eval-idrid" / "confusion_matrix.png")))

## Train the quality model on EyeQ
There is no external quality test set; the validation metrics in `metrics.jsonl` are the result.

In [ ]:
!python scripts/train.py --config configs/train_quality.yaml --override data_config={DATA}

In [ ]:
quality_run = max(Path("runs").glob("eyeq-*"), key=lambda path: path.stat().st_mtime)
best = max(
    (json.loads(line) for line in (quality_run / "metrics.jsonl").read_text().splitlines()),
    key=lambda record: record["qwk"] or -1,
)
for name, value in best.items():
    print(f"{name:40s} {value}")
print(quality_run / "best.pt")

## Keep only the results
Kaggle saves everything under `/kaggle/working` as output. The image cache is
already in `/tmp`; dropping each run's `last.pt` leaves the files the app needs.


In [ ]:
for path in Path("runs").rglob("last.pt"):
    path.unlink()